# Rabi Oscillation Experiment Prototype Simulation
Using a frequency mixer on both QCM-RM outputs, in order to view the pulse on the oscilloscope

Simulates rabi oscillation with amplitude of the voltage jump being a*cos(wt), where w is some frequecy I set, 
and t is the pulse duration, which I will loop in an external loop. I will change gate 1 to be the voltage jump, and leave all the other outputs the same.

The acquisition data for each outer loop in the simulation is then saved in the "Simulation data" folder, where there is also a notebook to generate plots from this data.

Connect:
- QCM output 0 to QRM input 0
- QCM output 1 to oscilloscope
- QCM-RF both outputs to frequency mixer
- Frequency mixer output to oscilloscope


Author: Kyle MacRobbie

---
---

## Expeiment sequence summary

|Step|Physical Description|Gate 1|Gate 2|QCM-RF output|
|-|-|-|-|-|
|1|Load electron into (1,0) state|Constant at -0.325 V|Constant at -0.25 V|None|
|2|Load electron into (2,0)S state|Constant at -0.395 V|Constant at -0.25 V|None|
|3|Tunnel electron into (1,1)S state|Ramp from -0.395 V to -0.325 V|Ramp from -0.25 V to -0.325 V|None|
|4|Apply RF pulse|Constant at -0.325 V|Constant at -0.325 V|RF pulse|
|5|Readout|Ramp from -0.325 V to -0.38 V|Ramp from -0.325 V to -0.27 V|None|
|6|Unload electrons|Ramp from -0.38 V to -0.325 V|Ramp from -0.27 V to -0.25 V|None|

<br>

---
---

### Setup

In [1]:
# Imports
import matplotlib.pyplot as plt # type: ignore
import pyvisa # type: ignore
import math
import numpy as np # type: ignore
from numpy import random # type: ignore
from __future__ import annotations 
from typing import TYPE_CHECKING, Callable
from qcodes.instrument import find_or_create_instrument # type: ignore
from qblox_instruments import Cluster, ClusterType # type: ignore
if TYPE_CHECKING:
    from qblox_instruments.qcodes_drivers.module import Module # type: ignore

In [2]:
# Run to get cluster IP
!qblox-pnp list

Devices:
 - 129.97.9.55: cluster_mm 0.10.0 with name "cluster-mm" and serial number 00015_2251_003


In [3]:
# Connect to cluster
cluster_ip = "129.97.9.55"
cluster_name = "cluster-mm"
cluster = find_or_create_instrument(
    Cluster,
    recreate=True,
    name=cluster_name,
    identifier=cluster_ip,
    dummy_cfg=(
        {
            2: ClusterType.CLUSTER_QCM,
            4: ClusterType.CLUSTER_QRM,
            6: ClusterType.CLUSTER_QCM_RF,
        }
        if cluster_ip is None
        else None
    ),
)
cluster.led_brightness('medium') # Sets LED brightness on the modules. Options are 'low', 'medium' and 'high'
def get_connected_modules(cluster: Cluster, filter_fn: Callable | None = None) -> dict[int, Module]:
    def checked_filter_fn(mod: ClusterType) -> bool:
        if filter_fn is not None:
            return filter_fn(mod)
        return True

    return {
        mod.slot_idx: mod for mod in cluster.modules if mod.present() and checked_filter_fn(mod)
    }
modules = get_connected_modules(cluster)
module = list(modules.values())[0]
cluster.led_brightness('medium')
cluster.reset()
print(cluster.get_system_status())

c:\Users\BaughLaflamme\anaconda3\envs\zh-test\lib\site-packages\qcodes\instrument\instrument_base.py:617: UserWarning: Changed cluster-mm to cluster_mm for instrument identifier
  warnings.warn(f"Changed {name} to {new_name} for instrument identifier")


Status: OKAY, Flags: NONE, Slot flags: NONE


In [4]:
# Define which modules will be used
print(modules)
qcm_module = modules[2]
qrm_module = modules[4]
rf_module = modules[6]

{2: <Module: cluster_mm_module2 of Cluster: cluster_mm>, 4: <Module: cluster_mm_module4 of Cluster: cluster_mm>, 6: <Module: cluster_mm_module6 of Cluster: cluster_mm>}


In [5]:
# Check QCM
print("\nQCM: {}\nQRM: {}\nRF: {}".format(qcm_module.is_qcm_type, qcm_module.is_qrm_type, qcm_module.is_rf_type))


QCM: True
QRM: False
RF: False


In [6]:
# Check QCM-RF
print("\nQCM: {}\nQRM: {}\nRF: {}".format(rf_module.is_qcm_type, rf_module.is_qrm_type, rf_module.is_rf_type))


QCM: True
QRM: False
RF: True


In [7]:
# Check QRM
print("\nQCM: {}\nQRM: {}\nRF: {}".format(qrm_module.is_qcm_type, qrm_module.is_qrm_type, qrm_module.is_rf_type))


QCM: False
QRM: True
RF: False


### Connecting to oscilloscope

In [8]:
# Get a list of all connected devices
#rm = pyvisa.ResourceManager()
#rm.list_resources()

In [9]:
# Connect to the oscillosocpe
#scope = rm.open_resource('GPIB0::8::INSTR')
#print(scope.query('*IDN?'))

### Global values that will remain constant through iterations

In [10]:
"""
With Qblox, when setting a voltage, you do not simply input the voltage you
would like to output.

For setting the offset, you use an integer between +/- 32768, which maps
linearly to outputs of +/- 1 V.
	For example, if you would like to output +0.5 V, your Q1ASM command
	would have to be:
		set_awg_offs	16384,16384
	since 16384 is 0.5 of 32768.

For playing waveforms, the values of the data points in that waveform can 
be in the range +/- 1, which corresponds to the fraction of the maximum
output of the module that will be outputted.
	For exampple, the QCM has a range of +/- 2.5 V, so playing a square
	waveform with amplitude 0.5 will output at a voltage of 0.5*2.5 V = 1.25 V.

Here we define these values as the voltages we wish to play, then account
for the input formats, which will then ultimately be used in the Q1ASM
sequence.
"""

qcm_range = 2.5				# +/- output voltage range of the QCM (5 V peak to peak)
awg_offs_range = 32768		# +/- range of Q1ASM arguements to the output offset on the QCM

# Offset voltage on the first output will be assigned in the rabi function, as it will change with different pulse durations.
# This is defined in the run_rabi_sequence function since it will need to change with different steps in the simulation.

offs21 = 0					# Offset for gate 2 step 1 
offs21q1 = round((offs21/qcm_range)*awg_offs_range)		# Converting to account for Q1ASM arguement range

ramp23_i = 0/qcm_range		# Start voltage for gate 2 step 3
ramp23_f = 1.0/qcm_range	# Final voltage for gate 2 step 3

offs24 = 1.0				# Offset for gate 2 step 4
offs24q1 = round((offs24/qcm_range)*awg_offs_range)		# Converting to account for Q1ASM arguement range

ramp25_i = 1.0/qcm_range	# Start voltage for gate 2 step 5
ramp25_f = 0.02/qcm_range	# Final voltage for gate 2 step 5

ramp26_i = 0.02/qcm_range	# Start voltage for gate 2 step 6
ramp26_f = 0/qcm_range		# Final voltage for gate 2 step 6

### End of setup

---
---

### Function to run the experiment for one iteration

In [11]:
def run_rabi_sequence(rf_pulse_length, rf_pulse_freq, gate_pulse_length, readout_length, ramp_length, num_iterations, mixer_cal = False):

	# MAKE WAVEFORMS

	waveforms_1 = {
		"ramp23" : {	# Ramp played on gate 2 in step 3
			"data": np.linspace(ramp23_i,ramp23_f,ramp_length).tolist(),
			"index": 23
		},
		"ramp256": {	# Ramps played on gate 2 in steps 5 and 6
			"data": np.linspace(ramp25_i,ramp25_f,100).tolist() + (np.zeros(readout_length - 100) + ramp25_f).tolist() + np.linspace(ramp26_i,ramp26_f,ramp_length).tolist(),
			"index": 256
		},
	}
	waveforms_rf = {
		"block": {		# Block to be modulated and played as the RF pulse in step 4.
			"data": [1.0 for i in range(5000)],
			"index": 0
		}
	}

	# MAKE ACQUISITION
	acquisitions = {
		"acq": {"num_bins": 1, "index": 0}
	}

	# The simulation pulse calculations
	qcm_range = 2.5
	awg_offs_range = 32767
	offs1 = 0.02*math.cos((6.3e5)*(rf_pulse_length)*(1e-9)) # Voltage to be set as the simulated pulse
	offs1q1 = round((offs1/qcm_range)*awg_offs_range)		# Converting to the corresponding Q1ASM value

	# MAKE SEQUENCES

	# Syncing with other sequencers and resetting the offest to 0 if it is not already 0
	seq_qcm0 = f"""
		  move {num_iterations},R0     # Loop iterator
	
	loop: 
		  wait_sync		  4
	
		  set_awg_offs	  0,0
		  upd_param		  {gate_pulse_length*2 + ramp_length + rf_pulse_length + round((readout_length/4))} # Wait for the readout step, and a little bit into the acquisition
	
		  set_awg_offs	  {offs1q1},{offs1q1}	# Set the voltage for the simulated readout pulse
		  upd_param       {round((3*readout_length/4)) + ramp_length}	# Acquire

		  set_awg_offs    0,0	# Reset voltage
		  upd_param		  4		# Apply the 0 V offset

	      loop            R0,@loop	# Loop

		  set_awg_offs	  0,0	# Ensure the voltage is set to 0 V before stopping
		  upd_param		  4		# Apply the 0 V offset
		  stop
	"""

	# Syncing with other sequencers and resetting the offest to 0 if it is not already 0
	seq_qcm1 = f"""
		  move {num_iterations},R0       # Loop iterator
	
	loop: 
		  wait_sync		  4
	
		  set_awg_offs	  {offs21q1},{offs21q1}	# Set the offset to the first step
		  upd_param		  {gate_pulse_length}	# Update offset and wait for the duration of the first step
	
		  wait			  {gate_pulse_length}	# Wait for the duration of the second step
	
		  play			  23,23,{ramp_length}	# Play the ramp for the third step
		  set_awg_offs	  {offs24q1},{offs24q1}	# Set the offset to the fourth step
		  upd_param		  {rf_pulse_length}		# Update offset and wait for the duration of the fourth step

		  set_awg_offs    {offs21q1},{offs21q1} # Set the offset for the last steps to be played over
		  play			  256,256,{ramp_length + readout_length}	# Play the last ramps, with a wait time between for readout
	
		  loop            R0,@loop	# Loop

		  set_awg_offs	  0,0	# Ensure the voltage is set to 0 V before stopping
		  upd_param		  4		# Apply the 0 V offset
		  stop
	"""

	# RF sequence, also sets marker for viewing on the oscilloscope
	seq_rf = f"""
		  move {num_iterations},R0     #Loop iterator
	
	loop: 
		  reset_ph			# Reset phase
		  wait_sync	  4		# Sync with other sequencers
		  set_mrk	  {0b1111}	# Turn on the marker and enable both RF outputs
		  upd_param	  {gate_pulse_length*2 + ramp_length}	# Wait until step 4

		  set_ph	  0		# Ensure phase is set to 0 before playing
		  play		  0,0,{rf_pulse_length}	# Play the RF pulse
		  
		  wait 		  {ramp_length + readout_length}	# Wait for the rest of the sequence
		
		  set_mrk	  {0b0000}	# Turn off the marker and disable the RF outputs
		  upd_param	  4			# Apply the marker changes
		  loop        R0,@loop	# Loop
		  stop
	"""

	# Reference 3 GHz so that we can use the frequency mixer and actually see the results on the oscilloscope
	seq_reference = f"""
		  move        {num_iterations},R0     # Loop iterator
	
	loop: 
		  reset_ph			# Reset the marker
		  wait_sync	  4		# Sync with the other sequencers
		  wait		  {gate_pulse_length*2 + ramp_length}	# Wait until step 4

		  set_ph	  0		# Ensure the phase is set to 0 before playing
		  play		  0,0,{rf_pulse_length}	# Play the RF pulse
		  
	   	  wait		  {ramp_length + readout_length}	# Wait for the rest of the sequence

		  loop        R0,@loop	# Loop
		  stop
	"""

	# Readout sequence
	seq_readout = f"""
		  move        {num_iterations},R0     # Loop iterator
	
	loop: 
		  wait_sync	  4		# Sync with the other sequencers
		  wait		  {gate_pulse_length*2 + ramp_length + rf_pulse_length}	  # Wait until step 4

		  acquire	  0,0,{readout_length}	# Acquire during the readout step
		  
		  wait		  {ramp_length}	# Wait the rest of the sequence

	      loop        R0,@loop	# Loop
		  stop
	"""

	# Create sequence dictionaries
	sequence_qcm0 = {
		"waveforms": {},
		"weights": {},
		"acquisitions": {},
		"program": seq_qcm0,
	}
	sequence_qcm1 = {
		"waveforms": waveforms_1,
		"weights": {},
		"acquisitions": {},
		"program": seq_qcm1,
	}
	sequence_rf = {
		"waveforms": waveforms_rf,
		"weights": {},
		"acquisitions": {},
		"program": seq_rf,
	}
	sequence_reference = {
		"waveforms": waveforms_rf,
		"weights": {},
		"acquisitions": {},
		"program": seq_reference,
	}
	sequence_readout = {
		"waveforms": {},
		"weights": {},
		"acquisitions": acquisitions,
		"program": seq_readout,
	}

	# Upload sequences to their respective sequencers
	qcm_module.sequencer0.sequence(sequence_qcm0)
	qcm_module.sequencer1.sequence(sequence_qcm1)
	rf_module.sequencer0.sequence(sequence_rf)
	rf_module.sequencer1.sequence(sequence_reference)
	qrm_module.sequencer0.sequence(sequence_readout)

	# Disconnect previous output connections outputs
	qcm_module.disconnect_outputs()
	rf_module.disconnect_outputs()
	qrm_module.disconnect_outputs()
	qrm_module.disconnect_inputs()

	# Connect outputs and inputs
	qcm_module.sequencer0.connect_out0("I")
	qcm_module.sequencer1.connect_out1("I")
	rf_module.sequencer0.connect_out0(True)
	rf_module.sequencer1.connect_out1(True)
	qrm_module.sequencer0.connect_acq_I("in0") # Acquire through first input

	qrm_module.scope_acq_sequencer_select(0) # Configure scope mode
	qrm_module.scope_acq_trigger_mode_path0("sequencer") # Trigger the acquisition with the sequencer

	qrm_module.scope_acq_avg_mode_en_path0(True) # Enable averaging over many loops

	# Enable NCO and LO modulation and set their respective frequencies for output 0
	rf_module.sequencer0.mod_en_awg(True)
	rf_module.out0_lo_en(True)
	rf_module.sequencer0.nco_freq(100e6 + rf_pulse_freq)
	rf_module.out0_lo_freq(3e9)

	# Enable NCO and LO modulation and set their respective frequencies for output 1
	rf_module.sequencer1.mod_en_awg(True)
	rf_module.out1_lo_en(True)
	rf_module.sequencer1.nco_freq(100e6)
	rf_module.out1_lo_freq(3e9)

	# Mixer calibration, if it is chosen
	if mixer_cal == True:
		rf_module.out0_lo_cal()
		rf_module.out1_lo_cal()
		rf_module.sequencer0.sideband_cal()
		rf_module.sequencer1.sideband_cal()

	qcm_module.sequencer0.sync_en(True)	# Enable sync
	qcm_module.sequencer1.sync_en(True)	# Enable sync
	rf_module.sequencer0.sync_en(True)	# Enable sync
	rf_module.sequencer1.sync_en(True)	# Enable sync
	qrm_module.sequencer0.sync_en(True)	# Enable sync

	# Arm sequencers
	qcm_module.arm_sequencer(0)
	qcm_module.arm_sequencer(1)
	rf_module.arm_sequencer(0)
	rf_module.arm_sequencer(1)
	qrm_module.arm_sequencer(0)

	cluster.start_sequencer()	# Run the sequence

	# Stop sequencers
	qcm_module.stop_sequencer(0)
	qrm_module.stop_sequencer(0)
	rf_module.stop_sequencer(0)
	qcm_module.stop_sequencer(1)
	rf_module.stop_sequencer(1)

	# Get the acquisition data
	qrm_module.get_acquisition_status(0) # Wait for the sequencer to stop with a timeout period of one minute.
	qrm_module.store_scope_acquisition(0, 'acq') # Move acquisition data from temporary memory to acquisition list.
	data = qrm_module.get_acquisitions(0) # Get acquisition list from instrument.

	# Print the sequencer status
	print("QCM sequencer 0:    " + str(qcm_module.get_sequencer_status(0)))
	print("QCM sequencer 1:    " + str(qcm_module.get_sequencer_status(1)))
	print("QCM-RF sequencer 0: " + str(rf_module.get_sequencer_status(0)))
	print("QCM-RF sequencer 1: " + str(rf_module.get_sequencer_status(1)))
	print("QRM sequencer 0:    " + str(qrm_module.get_sequencer_status(0)))

	return data

### Reset and configure oscilloscope

In [12]:
""" scope.write(f'TIME_DIV {2e-6} S')		# Set the time scale on all oscilloscope channels
scope.write(f'C1:VOLT_DIV {0.2} V')		# Set the voltage scale on channel 1
scope.write(f'C2:VOLT_DIV {0.05} V')	# Set the voltage scale on channel 2
scope.write(f'C3:VOLT_DIV {0.12} V')	# Set the voltage scale on channel 3
scope.write(f'C4:VOLT_DIV {2.0} V')		# Set the voltage scale on channel 4
scope.write(f'TRIG_DELAY {-15e-6}')		# Set the time scale left/right on the display
scope.write(f'TRIG_MODE SINGLE')		# Set the trigger mode to "single" in order to take one acquisition """

' scope.write(f\'TIME_DIV {2e-6} S\')\t\t# Set the time scale on all oscilloscope channels\nscope.write(f\'C1:VOLT_DIV {0.2} V\')\t\t# Set the voltage scale on channel 1\nscope.write(f\'C2:VOLT_DIV {0.05} V\')\t# Set the voltage scale on channel 2\nscope.write(f\'C3:VOLT_DIV {0.12} V\')\t# Set the voltage scale on channel 3\nscope.write(f\'C4:VOLT_DIV {2.0} V\')\t\t# Set the voltage scale on channel 4\nscope.write(f\'TRIG_DELAY {-15e-6}\')\t\t# Set the time scale left/right on the display\nscope.write(f\'TRIG_MODE SINGLE\')\t\t# Set the trigger mode to "single" in order to take one acquisition '

### Set simulation parameters and run the simulation, saving data to text files

In [50]:
rf_pulse_length_ = 200	
rf_pulse_freq_ = 100e6			# Hz	# Frequency of the RF pulse (step 4)
gate_pulse_length_ = 500		# ns	# Length of the gate pulses (all steps except 4). For now they are all set to the same length
readout_length_ = 100			# ns
ramp_length_ = 100				# ns
num_iterations = 1			# Number of iterations for each pulse duration, averaged

In [51]:
readout_data = run_rabi_sequence(rf_pulse_length_, rf_pulse_freq_, gate_pulse_length_, readout_length_, ramp_length_, num_iterations)

	

QCM sequencer 0:    Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []
QCM sequencer 1:    Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []
QCM-RF sequencer 0: Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []
QCM-RF sequencer 1: Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []
QRM sequencer 0:    Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, ACQ_SCOPE_DONE_PATH_0, ACQ_SCOPE_DONE_PATH_1, ACQ_BINNING_DONE, Warning Flags: NONE, Error Flags: NONE, Log: []


### Stopping

In [15]:
qcm_module.stop_sequencer(0)
qrm_module.stop_sequencer(0)
rf_module.stop_sequencer(0)
qcm_module.stop_sequencer(1)
rf_module.stop_sequencer(1)

print("QCM sequencer 0:    " + str(qcm_module.get_sequencer_status(0)))
print("QCM sequencer 1:    " + str(qcm_module.get_sequencer_status(1)))
print("QCM-RF sequencer 0: " + str(rf_module.get_sequencer_status(0)))
print("QCM-RF sequencer 1: " + str(rf_module.get_sequencer_status(1)))
print("QRM sequencer 0:    " + str(qrm_module.get_sequencer_status(0)))

# Reset the cluster
cluster.reset()
print(cluster.get_system_status())

QCM sequencer 0:    Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []
QCM sequencer 1:    Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []
QCM-RF sequencer 0: Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []
QCM-RF sequencer 1: Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, Warning Flags: NONE, Error Flags: NONE, Log: []
QRM sequencer 0:    Status: OKAY, State: STOPPED, Info Flags: FORCED_STOP, ACQ_SCOPE_DONE_PATH_0, ACQ_SCOPE_DONE_PATH_1, ACQ_BINNING_DONE, Warning Flags: NONE, Error Flags: NONE, Log: []
Status: OKAY, Flags: NONE, Slot flags: NONE
